# Load datasets into MongoDB

Start with one file, then Python documents, multiple files and Hugging Face datasets.
Install BONDI with `uv sync` and start Docker.
This notebook creates `notebook-datasets` on port 27126. Change them if already in use.
The last cell deletes its data and configuration. Run cleanup if execution stops early.

In [ ]:
from pathlib import Path

from bondi.data import load_documents, load_file
from bondi.deployment import deploy_mongodb
from bondi.mongodb import MongoCollectionRef, MongoExecutor, MongoFindQuery

deployment = deploy_mongodb(name="notebook-datasets", port=27126)

## Load one file

Load `dev/business.json` into exactly `dataset_demo.business`, using `business_id` as the ID.

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "dev" / "business.json").is_file():
    repo_root = repo_root.parent
business_file = repo_root / "dev" / "business.json"
assert business_file.is_file(), "Run from the repository root or notebooks directory."

business_ref = MongoCollectionRef("dataset_demo", "business")
result = load_file(
    business_file, target=deployment, collection=business_ref, id_field="business_id"
)
print(result.destinations)
assert result.destinations == {"dataset_demo.business": 8}

## Inspect the collection

Read the inserted businesses with the same collection reference.

In [ ]:
with MongoExecutor.from_uri(deployment.uri) as executor:
    businesses = executor.run(MongoFindQuery(business_ref, {}), collect_results=True).documents
for document in businesses:
    print(document["_id"], document["name"])
assert len(businesses) == 8

## Load Python documents

Use existing IDs, or supply a namespace to generate missing IDs.

In [ ]:
result = load_documents(
    [{"_id": "one", "name": "Harbour Cafe"}, {"_id": "two", "name": "Garden Books"}],
    target=deployment,
    collection=MongoCollectionRef("dataset_demo", "shops"),
)
print(result.destinations)

generated = load_documents(
    [{"city": "Sydney"}, {"city": "Melbourne"}],
    target=deployment,
    collection=MongoCollectionRef("dataset_demo", "cities"),
    namespace="notebook-cities-v1",
)
print(generated.destinations)

## Reload explicitly

Create refuses nonempty collections. Replace clears the selected collection and loads it again.

In [ ]:
from bondi.data import LoadDataError

try:
    load_file(business_file, target=deployment, collection=business_ref, id_field="business_id")
except LoadDataError as error:
    print("Expected refusal:", error)

result = load_file(
    business_file,
    target=deployment,
    collection=business_ref,
    id_field="business_id",
    mode="replace",
)
assert result.destinations == {"dataset_demo.business": 8}

## Load multiple local files

Call the same function for each file; no manifest is needed. Extended JSON dates become BSON dates.

In [ ]:
import json
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory(prefix="bondi-dataset-demo-")
root = Path(workspace.name)
print(f"Temporary dataset directory: {root}")

sources = {
    "users": [{"_id": "u1", "name": "Ada"}],
    "posts": [{"_id": "p1", "user_id": "u1", "created": {"$date": "2026-01-01T00:00:00Z"}}],
}
for collection_name, documents in sources.items():
    path = root / f"{collection_name}.jsonl"
    path.write_text("".join(json.dumps(document) + "\n" for document in documents))
    result = load_file(
        path,
        target=deployment,
        collection=MongoCollectionRef("dataset_demo", collection_name),
        json_mode="extended",
        on_progress=lambda progress: print(progress.destinations),
    )

## Load one Hugging Face dataset

The following examples require network access and may download substantial data.
Set `RUN_HF` to True to run the single-dataset examples.
The source revision is pinned; the destination name stays exactly as specified.

In [ ]:
from bondi.data import load_hf_dataset, load_hf_repository

RUN_HF = False
HF_REPO = "origami-ml/jsynth-data"
HF_REVISION = "9af20c2698454dd33403c993501f5e1c0007643a"

if RUN_HF:
    adult = load_hf_dataset(
        HF_REPO,
        dataset="adult",
        split="train",
        revision=HF_REVISION,
        target=deployment,
        collection=MongoCollectionRef("dataset_demo", "adult"),
    )
    print(adult.destinations)

## Combine all splits

Omit split to combine all rows in one collection. No suffix or split field is added.

In [ ]:
if RUN_HF:
    combined = load_hf_dataset(
        HF_REPO,
        dataset="adult",
        revision=HF_REVISION,
        target=deployment,
        collection=MongoCollectionRef("dataset_demo", "adult_all"),
    )
    print(combined.destinations)
    print(combined.source_counts)

## Load every dataset in a Hub repository

This loads every configuration into a collection with the same name.
Enable it separately because it loads more than the Adult dataset.
All configurations use the same resolved revision. Completed loads remain if a later one fails.

In [ ]:
RUN_HF_REPOSITORY = False

if RUN_HF_REPOSITORY:
    results = load_hf_repository(
        HF_REPO, revision=HF_REVISION, target=deployment, database="hf_repository_demo"
    )
    for dataset_name, result in results.items():
        print(dataset_name, result.destinations)

## Remove the demo resources

Delete the deployment, all its data and configuration, and any temporary source files.

In [ ]:
deployment.remove()
if "workspace" in globals():
    workspace.cleanup()